<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [38]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    X, y = fetch_openml('mnist_784', version=1, as_frame=False, return_X_y=True)

    y = y.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    return X_train, X_test, y_train, y_test

O ideal é que seja sim normalizado para certos modelos como KNN, SVM etc, pois o dataset trabalha com pixels que vao de 0 a 255, ou seja, se colocarmos um valor entre 0 e 1, podemos impactar diretamente no desempenho do modelo.

Já para modelos como Árvores de decisao, Random Forest, a normalização não seria necessária, pois são modelos baseados em regras de divisão, ou seja, dividem em superficies de decisão baseado nos valores, sem fazer calculos diretamente com os valores do dataset.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
    )
    
    model.fit(X_train, y_train)
    
    return model

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(
        n_estimators=100, 
        random_state=seed
    )

    model.fit(X_train, y_train)
    
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [25]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    return acc

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [51]:
def run_pipeline(model_type, seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)

    match model_type:
      case "rf":
        model = train_random_forest(X_train, y_train, seed)

      case "ab":
        model = train_adaboost(X_train, y_train, seed)

    acc = evaluate(model, X_train, y_train)

    return acc

run_pipeline(model_type='rf', seed=42)

1.0

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

In [58]:
profundidades = [1,5,10,15,20,25,30,None]

seed = 42

X_train, X_test, y_train, y_test = load_data(seed)

for depth in profundidades:
    rf_model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        max_depth=depth
    )
    rf_model.fit(X_train, y_train)
    acc_treino = accuracy_score(y_train, rf_model.predict(X_train))
    acc_teste  = accuracy_score(y_test,  rf_model.predict(X_test))
    print(f"profundidade: {depth} | acurácia treino: {acc_treino:.4f} | acurácia treino: {acc_teste:.4f}")

profundidade: 1 | acurácia treino: 0.4969 | acurácia treino: 0.4986
profundidade: 5 | acurácia treino: 0.8609 | acurácia treino: 0.8600
profundidade: 10 | acurácia treino: 0.9666 | acurácia treino: 0.9454
profundidade: 15 | acurácia treino: 0.9977 | acurácia treino: 0.9626
profundidade: 20 | acurácia treino: 0.9993 | acurácia treino: 0.9669
profundidade: 25 | acurácia treino: 0.9998 | acurácia treino: 0.9680
profundidade: 30 | acurácia treino: 1.0000 | acurácia treino: 0.9677
profundidade: None | acurácia treino: 1.0000 | acurácia treino: 0.9672


Produndidade 30, pois é quando o modelo apresenta acurácia de treino 100% e uma acurácia de teste menor do que comparado à acurácia com profundidade 25.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [59]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_full(model, X_test, y_test):
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')

    return acc, precision, recall, f1

seed = 42

X_train, X_test, y_train, y_test = load_data(seed)

rf_model = train_random_forest(X_train, y_train, seed)
rf_metrics = evaluate_full(rf_model, X_test, y_test)

ab_model = train_adaboost(X_train, y_train, seed)
ab_metrics = evaluate_full(ab_model, X_test, y_test)

print("Random Forest:")
print(f"Acurácia:  {rf_metrics[0]:.4f}")
print(f"Precisão:  {rf_metrics[1]:.4f}")
print(f"Recall:    {rf_metrics[2]:.4f}")
print(f"F1-Score:  {rf_metrics[3]:.4f}\n")

print("AdaBoost:")
print(f"Acurácia:  {ab_metrics[0]:.4f}")
print(f"Precisão:  {ab_metrics[1]:.4f}")
print(f"Recall:    {ab_metrics[2]:.4f}")
print(f"F1-Score:  {ab_metrics[3]:.4f}")

Random Forest:
Acurácia:  0.9672
Precisão:  0.9670
Recall:    0.9669
F1-Score:  0.9669

AdaBoost:
Acurácia:  0.7387
Precisão:  0.7440
Recall:    0.7354
F1-Score:  0.7375


O modelo que apresentou desempenho inicial melhor foi o modelo de Random Fores, apresentando todas as métricas, como acurácia, precisão, recall e F1-score bem superiores ao modelo AdaBoost.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [52]:
acc_rf_model_seed_42 = run_pipeline(model_type='rf', seed=42)
acc_rf_model_seed_7 = run_pipeline(model_type='rf', seed=7)

print("Seed 42:")
print(f"Acurácia: {acc_rf_model_seed_42}")

print("Seed 7:")
print(f"Acurácia: {acc_rf_model_seed_7}")

Seed 42:
Acurácia: 1.0
Seed 7:
Acurácia: 1.0


Os resultados não mudaram, nem quando executado duas vezes, nem pela mudança da seed.

O resultado não mudou, mesmo mudando a seed, pelo fato de que o modelo está com max_depth como None, ou seja, está permitindo o overfitting do modelo, retornando uma acurácia de 100% para ambos os modelos executados com seeds diferentes.

Sim, o experimento é reprodutível. Pois o uso de random_state=seed garante que, para uma mesma seed, os resultados são sempre os mesmos.

Alterar a seed gera um novo experimento controlado, não um comportamento imprevisível

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [53]:
seed = 42

X_train, X_test, y_train, y_test = load_data(seed)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=seed,
)

rf_model.fit(X_train, y_train)

acc_treino = accuracy_score(y_train, rf_model.predict(X_train))
acc_teste  = accuracy_score(y_test,  rf_model.predict(X_test))

print(f"{acc_treino} {acc_teste:.4f}")

  1.0000   0.9672


Sim, existe overfitting. Pois a acurácia é 100% usando os dados de teste e a acurácia é de 96,72%. Isso indica que o modelo decorou os dados de treinamento mas errou algumas instâncias dos dados de teste.

Random Forest tende a sofrer mais overfitting se não controlado os hiper-parâmetros, ou seja, max_depth=None.

Isso causa o uso de várias árvores profundas, memorização de padrões específicos do treino, e
a grande de quantidade de superfícies de decisão devido a profundidade máxima do modelo

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [60]:
n_values = [10, 50, 100, 200]

seed = 42

X_train, X_test, y_train, y_test = load_data(seed)

print("=== Random Forest ===")
for n in n_values:
    model = train_random_forest(X_train, y_train, seed)
    model.set_params(n_estimators=n)
    model.fit(X_train, y_train)

    acc = evaluate(model, X_test, y_test)
    print(f"n_estimators={n}: {acc:.4f}")

print("\n=== AdaBoost ===")
for n in n_values:
    model = train_adaboost(X_train, y_train, seed)
    model.set_params(n_estimators=n)
    model.fit(X_train, y_train)

    acc = evaluate(model, X_test, y_test)
    print(f"n_estimators={n}: {acc:.4f}")

=== Random Forest ===
n_estimators=10: 0.9464
n_estimators=50: 0.9639
n_estimators=100: 0.9672
n_estimators=200: 0.9682

=== AdaBoost ===
n_estimators=10: 0.3337
n_estimators=50: 0.6681
n_estimators=100: 0.7387
n_estimators=200: 0.7684


O desempenho não muda de forma significativamente grande no caso do Random Forest, pois o aumento do número de estimadores gera apenas melhorias graduais que rapidamente se estabilizam. Já no AdaBoost, as mudanças são mais perceptíveis, podendo ocorrer tanto melhora quanto instabilidade conforme o número de estimadores aumenta.

Dessa forma, o AdaBoost é o modelo mais sensível a mudanças nesse hiperparâmetro, enquanto o Random Forest apresenta comportamento mais estável.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1.
  A acurácia não é suficiente, isoladamente, para avaliar os modelos. Embora seja uma métrica simples e útil, ela pode esconder problemas, especialmente em cenários com múltiplas classes ou possíveis desequilíbrios. Métricas como precisão, recall e F1-score complementam a análise, permitindo entender melhor o comportamento do modelo em cada classe e identificar possíveis erros relevantes que a acurácia não evidencia.

---

2.
  Para garantir que o resultado não ocorreu por acaso, é necessário controlar a aleatoriedade por meio do uso de random_state e repetir os experimentos com diferentes seeds. Dessa forma, é possível verificar a consistência dos resultados e reduzir a dependência de uma única divisão dos dados, tornando as conclusões mais confiáveis.

---

3.
  Dois possíveis problemas metodológicos neste experimento são a utilização de apenas uma divisão treino/teste, o que pode introduzir viés nos resultados, e a ausência de ajuste de hiperparâmetros, que pode levar a comparações injustas entre modelos. Além disso, não considerar múltiplas métricas também pode limitar a qualidade da avaliação.

---
4.
  O pipeline implementado é razoavelmente confiável, pois é estruturado, modular e reprodutível devido ao uso de seeds. No entanto, sua confiabilidade é limitada pelo fato de utilizar apenas uma divisão dos dados e não explorar validação cruzada ou tuning de hiperparâmetros, o que seria necessário para uma avaliação mais robusta.

  Além disso, o treinamento dos modelos a cada execução das funções deixa a avaliação dos modelos mais lenta. O ideal seria chamar a execução dos modelos de treinamento em uma celula específica, salvando os modelos em memória para uma análise mais rápida.